# YOLO_ChestXray: Master AI Training, Validation & Diagnostic Platform (Plan 1-3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wupinlai/YOLO_ChestXray/blob/main/notebooks/YOLO_ChestXray_Colab_Training.ipynb)

This notebook implements **Plan 1-3** standard with automatic YOLOv7 environment patching for PyTorch 2.x & NumPy 1.24+, non-interactive execution (WANDB disabled), reproducibility seeds, dataset integrity checks, strict stage-wise `--resume` training, 10-sample baseline inference with IoU, error diagnostics, 2x2 galleries, and 16-page master PDF report compilation.

## 1. Reproducibility & Environment Check (Plan 1-3)

In [ ]:
import os, random
import numpy as np
import torch

# 1. Non-Interactive Requirement (WANDB disabled)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

# 2. NumPy Legacy Type Alias Patch
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

# 3. Seed Locking for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

!nvidia-smi
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Google Drive Mounting & Kaggle Credentials

In [ ]:
from google.colab import drive
import os, json

drive.mount('/content/drive')

# Google Drive Structure per Plan 1-3
DRIVE_DIR = '/content/drive/MyDrive/YOLO_ChestXray'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/final_report", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/final_inference", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/releases", exist_ok=True)

# Load Kaggle Credential
token_path = '/content/drive/MyDrive/Secrets/kaggle_token.txt'
kaggle_token = None
if os.path.exists(token_path):
    with open(token_path, 'r') as f:
        kaggle_token = f.read().strip()
    print("Loaded Kaggle token from Google Drive Secrets!")
else:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get('KAGGLE_KEY') or userdata.get('KAGGLE_API_TOKEN')
        print("Loaded Kaggle token from Colab Secrets!")
    except Exception:
        pass

if kaggle_token:
    os.environ['KAGGLE_KEY'] = kaggle_token
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump({'key': kaggle_token}, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print("Kaggle API configured successfully.")
else:
    print("Please provide /content/drive/MyDrive/Secrets/kaggle_token.txt or set Colab Secret KAGGLE_KEY.")

## 3. Clone Repository & Setup Dependencies with Compatibility Fixes

In [ ]:
%cd /content
!git clone https://github.com/wupinlai/YOLO_ChestXray.git
%cd /content/YOLO_ChestXray

# Clone YOLOv7 official base for backbone models
!git clone https://github.com/WongKinYiu/yolov7.git yolov7_repo
!cp -r yolov7_repo/* ./ || true
!pip install -r requirements.txt

# Apply YOLOv7 PyTorch 2.x & NumPy Compatibility Fixes
!python scripts/fix_yolov7_env.py ./

# Download official pretrained YOLOv7 weights
!wget -nc https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

## 4. Download Kaggle Dataset & Integrity Check (Plan 1-3)

In [ ]:
!mkdir -p datasets/chestxray8
!kaggle datasets download -d spritan1/yolo-annotated-chestxray-8-object-detection -p datasets/
!unzip -q -o datasets/yolo-annotated-chestxray-8-object-detection.zip -d datasets/chestxray8/

# Dataset Integrity Verification
from scripts.dataset_utils import check_dataset_integrity
df_integrity = check_dataset_integrity(
    'datasets/chestxray8/train/images', 'datasets/chestxray8/train/labels',
    'datasets/chestxray8/val/images', 'datasets/chestxray8/val/labels',
    'reports/dataset_integrity_report.csv'
)
print(df_integrity.head(10))

## 5. Dataset Configuration & Statistical Analysis

In [ ]:
import yaml
from scripts.dataset_utils import run_dataset_analysis

config = {
    'train': '/content/YOLO_ChestXray/datasets/chestxray8/train/images',
    'val': '/content/YOLO_ChestXray/datasets/chestxray8/val/images',
    'nc': 14,
    'names': [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
        'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
        'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
    ]
}
os.makedirs('configs', exist_ok=True)
with open('configs/chestxray.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

# Run dataset statistical analysis
run_dataset_analysis(
    'configs/chestxray.yaml',
    'datasets/chestxray8/train/images', 'datasets/chestxray8/train/labels',
    'datasets/chestxray8/val/images', 'datasets/chestxray8/val/labels',
    'reports/dataset_analysis'
)

## 6. Pre-training: Select 10 Random Baseline Samples (Plan 1-3)

In [ ]:
!python scripts/inference.py --action sample --val-dir datasets/chestxray8/val/images --output-dir reports/final_inference --num-samples 10
!cp -r reports/final_inference/* {DRIVE_DIR}/final_inference/ || true

## 7. Stage-wise Training Pipeline (Strict Plan 1-3 Resume Standard)

In [ ]:
# Stage 1: Epochs 1 ~ 10
!python scripts/train.py --weights yolov7.pt --stage 1 --epochs 10 --batch-size 16 --name stage_1
!cp checkpoints/checkpoint_10.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 2: Epochs 11 ~ 20 (Resume from checkpoint_10.pt)
!python scripts/train.py --resume checkpoints/checkpoint_10.pt --stage 2 --epochs 20 --name stage_2
!cp checkpoints/checkpoint_20.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 3: Epochs 21 ~ 30 (Resume from checkpoint_20.pt)
!python scripts/train.py --resume checkpoints/checkpoint_20.pt --stage 3 --epochs 30 --name stage_3
!cp checkpoints/checkpoint_30.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 4: Epochs 31 ~ 40 (Resume from checkpoint_30.pt)
!python scripts/train.py --resume checkpoints/checkpoint_30.pt --stage 4 --epochs 40 --name stage_4
!cp checkpoints/checkpoint_40.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 5: Epochs 41 ~ 50 (Resume from checkpoint_40.pt)
!python scripts/train.py --resume checkpoints/checkpoint_40.pt --stage 5 --epochs 50 --name stage_5
!cp checkpoints/checkpoint_50.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

## 8. Validation on Best Model Checkpoint

In [ ]:
!python scripts/validate.py --weights checkpoints/best_model.pt --data configs/chestxray.yaml --name val_results
!cp reports/*.png {DRIVE_DIR}/reports/ || true
!cp reports/*.csv {DRIVE_DIR}/reports/ || true

## 9. Comprehensive Plan 1-3 Post-Processing, Diagnostics & 16-Page Master PDF Report

In [ ]:
!python scripts/run_plan1_pipeline.py

# Sync all reports and final PDF to Google Drive
!cp -r reports/* {DRIVE_DIR}/reports/
!cp reports/final_report/final_report.pdf {DRIVE_DIR}/final_report/final_report.pdf
print("All Plan 1-3 deliverables & 16-page master PDF synced to Google Drive!")

## 10. Display Visual Results (Inference & Error Galleries)

In [ ]:
from IPython.display import Image, display

print("--- Final Multi-Sample Inference Result (Sample 1) ---")
if os.path.exists('reports/final_inference/result_01.jpg'):
    display(Image(filename='reports/final_inference/result_01.jpg', width=600))

print("--- Error Gallery (2x2 Grid) ---")
if os.path.exists('reports/error_gallery.jpg'):
    display(Image(filename='reports/error_gallery.jpg', width=700))

print("--- Confidence Threshold Sweep Analysis ---")
if os.path.exists('reports/confidence_threshold_analysis.png'):
    display(Image(filename='reports/confidence_threshold_analysis.png', width=600))

## 11. Package Release Assets for GitHub Release

In [ ]:
!zip -r {DRIVE_DIR}/releases/Release_50_Plan1_3_assets.zip \
  checkpoints/checkpoint_50.pt \
  checkpoints/best_model.pt \
  reports/final_report/final_report.pdf \
  reports/dataset_integrity_report.csv \
  reports/project_manifest.json \
  reports/*.png \
  reports/*.csv

print(f"Plan 1-3 Release bundle created at {DRIVE_DIR}/releases/Release_50_Plan1_3_assets.zip")